In [75]:
import numpy as np
import pandas as pd
import invert4geom

import xarray as xr

In [76]:
#from polartoolkit import fetch, maps, profiles, regions, utils

import polartoolkit as ptk

In [177]:
import os

os.environ["POLARTOOLKIT_HEMISPHERE"] = "south"

region = ptk.regions.getz_ice_shelf
spacing = 4000

# 1.) Loading Datasets

In [178]:
constraint_points = pd.read_csv('constraint_points.csv')
constraint_points

,easting,northing,bed_elevation
0,-1.559078e+06,-1.233548e+06,-3538.024543
1,-1.558187e+06,-1.233347e+06,-3531.229456
2,-1.557547e+06,-1.233500e+06,-3520.570221
3,-1.556466e+06,-1.233429e+06,-3513.796082
4,-1.555333e+06,-1.233868e+06,-3512.433386
...,...,...,...
55221,-1.154000e+06,-6.640000e+05,-957.328003
55222,-1.152750e+06,-6.645000e+05,-961.495514
55223,-1.147000e+06,-6.640000e+05,-362.992767
55224,-1.136500e+06,-6.645000e+05,-96.594223


In [180]:
bed_topography = ptk.fetch.bedmap3(layer="bed", region = region, reference="ellipsoid",spacing=spacing)

bed_topography = bed_topography.rename({'x': 'easting', 'y': 'northing'}).to_dataset(name = 'upward')

In [ ]:
bed_topography

In [ ]:
gravityanomaly = xr.load_dataset('gravityanomaly.nc')
gravityanomaly

In [ ]:
gravityanomaly = xr.load_dataset('gravityanomaly.nc')

resampled_vars = []
for var in gravityanomaly.data_vars:
    resampled_vars.append(
        ptk.resample_grid(gravityanomaly[var], spacing=spacing).rename(var)
    )
gravityanomaly = xr.merge(resampled_vars).rename({'x': 'easting', 'y': 'northing'})
gravityanomaly

# 2.) Creating data instance

In [ ]:
gravityanomaly = invert4geom.create_data(gravityanomaly, buffer_width=None, model_type='prisms')
gravityanomaly

In [ ]:
gravityanomaly.inv.plot_observed()

In [ ]:
water_surface = ptk.fetch.bedmap3(layer = 'icebase', region = region, reference="ellipsoid",spacing = spacing)
water_surface = water_surface.rename( {'x': 'easting', 'y': 'northing'})


In [ ]:
model = invert4geom.create_model(
     zref = 0,
     density_contrast= 2700 - 1030, topography = bed_topography, upper_confining_layer=water_surface

)
model

In [ ]:
model.inv.plot_model(zscale = 20, color_by = 'density')

# 4.) calculated gravity effect of starting model

In [ ]:
gravityanomaly.inv.forward_gravity(model, progressbar=True)
gravityanomaly


# 5.) Gravity misfit

In [ ]:
# in many cases, we want to remove a regional signal from the misfit to isolate the
# residual signal. In this simple case, we assume there is no regional misfit and set
# it to 0
gravityanomaly.inv.regional_separation(
   constraints_df = constraint_points,
    method="constraints",
    tension_factor=0.3,
    grid_method= 'pygmt',
)
gravityanomaly

In [ ]:
gravityanomaly.inv.plot_anomalies(coast=True, points=constraint_points, points_style='p.2p')

# 7.) Run Inversion

In [ ]:
# setup the inversion
invs = invert4geom.Inversion(
    gravityanomaly,
    model,
    solver_damping=0.05,        #normal = 0.05
    # set stopping criteria
    max_iterations=30,
    l2_norm_tolerance=2,  # stop if L2-norm < 2 mGal (RMSE of 4 mGal)
    delta_l2_norm_tolerance=1.01,  # stop if iteration's change in L2-norm < 1%
)
invs.__dict__

In [ ]:
invs.invert(plot_dynamic_convergence=True)


In [ ]:
invs.termination_reason

In [ ]:
invs.stats_df

In [ ]:
invs.model

In [ ]:
invs.plot_inversion_results(
    coast=True,
)

In [ ]:
invs.model.topography.plot()

# 8.) Comparision

In [ ]:
bed_topography2 = ptk.fetch.bedmap2(layer="bed", region=region, reference="ellipsoid", spacing=spacing)


bed_topography2 = bed_topography2.rename({'x': 'easting', 'y': 'northing'}).to_dataset(name='upward')

In [ ]:
bed_topography2

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2,ax3) = plt.subplots(1, 3, figsize=(12, 5))


invs.model.topography.plot(ax = ax1)

bed_topography.upward.plot.pcolormesh(x='easting', y='northing', ax=ax2, robust=True)


bed_topography2.upward.plot.pcolormesh(x='easting', y='northing', ax=ax3, robust=True)


In [ ]:
bed_topography2

In [ ]:
diff1 = invs.model.topography - bed_topography

diff2 = invs.model.topography - bed_topography2

diff2


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

diff1.upward.plot.pcolormesh(ax=ax1)
ax1.set_title('Difference final Topography - Bedmap3')
diff2.upward.plot.pcolormesh(ax=ax2)
ax2.set_title('Difference final Topography - Bedmap2')